# RSNA 2024 Lumbar Spine — Preprocessing Verification
Purpose: verify the processed dataset, manifest, label files, fold splits, and class weights before launching training.

In [ ]:
# !pip install ultralytics rich -q

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/rsna-lumbar-yolo')
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
PROCESSED_ROOT = Path('/kaggle/input/rsna-lumbar-processed')
SPLITS_DIR = Path('/kaggle/input/rsna-lumbar-processed/splits')

In [ ]:
from src.preprocess.verify_dataset import run_all_checks
passed = run_all_checks(PROCESSED_ROOT, SPLITS_DIR)
assert passed, "Dataset verification FAILED — fix before proceeding"
print("✓ All critical checks passed")

In [ ]:
manifest_df = pd.read_csv(PROCESSED_ROOT / 'manifest.csv')
print('Manifest shape:', manifest_df.shape)
display(manifest_df.groupby('series_type').size().rename('count'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
manifest_df['original_rows'].hist(bins=30, ax=ax[0]); ax[0].set_title('Original rows distribution')
manifest_df['original_cols'].hist(bins=30, ax=ax[1]); ax[1].set_title('Original cols distribution')
plt.tight_layout(); plt.show()

In [ ]:
labels_df = pd.read_csv(PROCESSED_ROOT / 'labels_summary.csv')
print('Total annotations:', labels_df['n_annotations'].sum())
display(labels_df.groupby('series_type')['n_annotations'].sum())
labels_df['n_annotations'].hist(bins=30)
plt.xlabel('Annotations per image'); plt.tight_layout(); plt.show()

In [ ]:
import cv2, random; random.seed(42)
from src.utils.bbox_utils import yolo_to_pixel, CLASS_NAMES
sample_studies = random.sample(list(manifest_df['study_id'].unique()), 3)
# For each study, load one PNG and draw its label boxes
for study in sample_studies:
    rows = manifest_df[manifest_df['study_id'] == study]
    fig, axes = plt.subplots(1, min(len(rows), 3), figsize=(12, 4))
    if len(rows) == 1: axes = [axes]
    for ax, (_, row) in zip(axes, rows.iterrows()):
        img = cv2.imread(str(PROCESSED_ROOT / row['output_path']), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        # Load label file
        label_path = str(row['output_path']).replace('images','labels').replace('.png','.txt')
        label_file = PROCESSED_ROOT / label_path
        if label_file.exists():
            for line in open(label_file):
                parts = line.split(); cls_id = int(parts[0]); x,y,w,h = map(float,parts[1:])
                x1,y1,x2,y2 = yolo_to_pixel(x,y,w,h,640,640)
                cv2.rectangle(img_rgb,(x1,y1),(x2,y2),(255,0,0),2)
        ax.imshow(img_rgb); ax.set_title(f"{row['series_type']}\n{row['study_id']}", fontsize=8); ax.axis('off')
    plt.suptitle(f"Study {study}"); plt.tight_layout(); plt.show()

In [ ]:
fold_dfs = []
for k in range(5):
    df = pd.read_csv(SPLITS_DIR / f'fold_{k}.csv')
    df['fold_file'] = k
    fold_dfs.append(df)
summary = []
for k, df in enumerate(fold_dfs):
    n_train = (df['split']=='train').sum(); n_val = (df['split']=='val').sum()
    summary.append({'fold': k, 'train': n_train, 'val': n_val})
display(pd.DataFrame(summary))
print("No overlap check:", "PASS" if all(
    len(set(d[d.split=='train'].study_id) & set(d[d.split=='val'].study_id))==0
    for d in fold_dfs) else "FAIL")

In [ ]:
import json
weights = json.load(open(PROCESSED_ROOT / 'class_weights.json'))
from src.utils.bbox_utils import CLASS_NAMES
for k, v in sorted(weights.items()):
    print(f"  {int(k):2d} {CLASS_NAMES[int(k)]:45s} {v:.4f}")
plt.bar(range(15), [weights[str(k)] for k in range(15)])
plt.xticks(range(15), [cn.split('_')[-1] for cn in CLASS_NAMES], rotation=90)
plt.title('Class weights'); plt.tight_layout(); plt.show()

In [ ]:
assert passed, "Dataset verification failed"
print("✓ Dataset verified — safe to proceed to training")
print("02_preprocess_verify.ipynb Complete ✓")